# 02d (v2, disk-space fix) · Sensitivity runs, seeds 1 and 2 (learning-rate schedule on the training loss)
**Runtime: T4 GPU.** Run all. Identical to notebook 02c (same code, data, folds, settings and the same single change:
ReduceLROnPlateau monitors the training loss); only the seeds differ (1 and 2 instead of 0).
Writes into the same folder as 02c, `results/p2_cbis_runs_e200_lrtrain`; seed-0 runs and the main runs are never touched.

60 runs (6 G/P models × 5 folds × seeds 1 and 2), roughly 15–25 min each (about 15–25 GPU hours in total, several sessions).
Resumable: after a disconnect just **Run all** again; finished runs are skipped.
To use two Colab sessions at the same time, set `SEEDS = [1]` in one and `SEEDS = [2]` in the other (cell 4).

**Analysis plan (fixed before these runs):** the sensitivity contrasts are recomputed with three seeds per model using the
same hierarchical bootstrap over seeds and clusters as the main analysis (Holm over the same 12-contrast family). No new decision
rule is added; claims about the training schedule will be limited to directions that are resolved under both schedules.
Every run keeps its `best.weights.h5`, `history.csv`, `per_image.csv` and `summary.json`.

**v2:** cell 3c re-saves each run's weights without the optimizer state (~31 MB instead of ~95 MB) after a weight-by-weight check, so the 60 runs need about 2 GB of Drive space instead of about 6 GB.


In [ ]:
# 1 · Drive, paths
from google.colab import drive
drive.mount('/content/drive')
import os, sys, glob, importlib

PROJECT  = '/content/drive/MyDrive/quantum-placement'
DATA_DIR = f'{PROJECT}/data/processed/cbis_mass_256'
CODE_DIR = f'{PROJECT}/code'
RUNS_DIR = f'{PROJECT}/results/p2_cbis_runs_e200_lrtrain'   # sensitivity runs (new folder; main runs are never touched)
MAIN_DIR = f'{PROJECT}/results/p2_cbis_runs_e200'           # main runs, read only
TIME_DIR = f'{PROJECT}/results/_timing'
QSEG_DIR = '/content/drive/MyDrive/QSEG_FINAL/code'            # Paper 1 package (folder that contains qseg/)
if not os.path.exists(f'{QSEG_DIR}/qseg/quantum_sim.py'):
    hits = glob.glob('/content/drive/MyDrive/**/qseg/quantum_sim.py', recursive=True)
    assert hits, 'Paper 1 qseg package not found on Drive'
    QSEG_DIR = os.path.dirname(os.path.dirname(hits[0]))
for d in (CODE_DIR, RUNS_DIR, TIME_DIR): os.makedirs(d, exist_ok=True)
assert os.path.exists(f'{DATA_DIR}/folds.csv'), 'Run notebook 01 first'
for p in (QSEG_DIR, CODE_DIR):
    if p not in sys.path: sys.path.insert(0, p)
print('qseg :', QSEG_DIR); print('data :', DATA_DIR); print('runs :', RUNS_DIR)


In [ ]:
# 2 · Use the Paper 2 library already on Drive (written by notebook 02; not rewritten here)
import importlib, qseg2; importlib.reload(qseg2)
import tensorflow as tf, numpy as np, pandas as pd
print('qseg2 from:', qseg2.__file__)
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), 'No GPU: Runtime -> Change runtime type -> T4 GPU'


In [ ]:
# 3 · Data + shared config (Paper 1 settings)
from qseg.config import Config
cfg = Config(work_dir=RUNS_DIR, max_epochs=200)   # v3: up to 200 epochs; batch 16, Adam 1e-4, early stop patience 10 after epoch 15
X, Y, DF = qseg2.load_cbis(DATA_DIR)
print(X.shape, Y.shape, 'ROIs:', len(DF), '| patients:', DF.patient.nunique())
print(DF.groupby('fold').agg(rois=('roi_id','size'), patients=('patient','nunique')).T.to_string())


In [ ]:
# 3b · The ONLY change vs the main runs: the learning-rate schedule watches the TRAINING loss.
# In the main runs ReduceLROnPlateau watched val_loss; an early validation-loss spike (BatchNorm moving
# statistics still settling) cut the learning rate every 4 epochs, so patch-wise models and 'full' reached
# lr <= 1e-6 by epoch 30. Factor 0.5, patience 4, min_lr 1e-7 stay the same. Early stopping and the best
# checkpoint still use val_loss (start after epoch 15, patience 10), exactly as before.
_Plateau = tf.keras.callbacks.ReduceLROnPlateau
if not getattr(_Plateau, '_p2_train_loss', False):
    class TrainLossPlateau(_Plateau):
        _p2_train_loss = True
        def __init__(self, monitor='val_loss', **kw):
            super().__init__(monitor='loss', **kw)
    tf.keras.callbacks.ReduceLROnPlateau = TrainLossPlateau
assert qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor == 'loss'
print('LR schedule now monitors:', qseg2.tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=4).monitor)


In [ ]:
# 3c · Disk-space fix: Keras stores the Adam optimizer state inside best.weights.h5 (~95 MB per run).
# After each run, the weights are re-saved WITHOUT the optimizer state (~31 MB) and the new file is
# verified weight-by-weight against the original before it replaces it. Model weights are unchanged.
def shrink_weights(run):
    d = f'{RUNS_DIR}/{run}'; big = f'{d}/best.weights.h5'; small = f'{d}/best_small.weights.h5'
    if not os.path.exists(big) or os.path.exists(f'{d}/.shrunk'):
        return
    if os.path.getsize(big) < 60e6:
        open(f'{d}/.shrunk', 'w').close(); return
    name = run.split('_fold')[0]
    tf.keras.backend.clear_session()
    m1 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m1.load_weights(big)
    m1.save_weights(small)
    m2 = qseg2.build_p2_model(name, (cfg.image_size, cfg.image_size, cfg.in_channels), cfg.n_qubits, cfg.n_q_layers)
    m2.load_weights(small)
    w1, w2 = m1.get_weights(), m2.get_weights()
    assert len(w1) == len(w2) and all(np.array_equal(a, b) for a, b in zip(w1, w2)), f'{run}: weight check failed, original kept'
    os.replace(small, big)
    open(f'{d}/.shrunk', 'w').close()
    print(f'   {run}: weights re-saved without optimizer state ({os.path.getsize(big) / 1e6:.0f} MB)', flush=True)
for r in sorted(os.listdir(RUNS_DIR)):                 # also fixes runs that finished before this cell existed
    if os.path.exists(f'{RUNS_DIR}/{r}/summary.json') and '_seed0' not in r:
        shrink_weights(r)


In [ ]:
# 4 · Training: seeds 1 and 2, six G/P models, 5 folds = 60 runs (resumable). Order: seed, fold, model.
MODELS_TO_RUN = ['g_q', 'g_cl', 'g_id', 'p_q', 'p_cl', 'p_id']
FOLDS, SEEDS  = [0, 1, 2, 3, 4], [1, 2]      # set [1] or [2] to split the work over two sessions
DATA = (X, Y, DF)
N = len(SEEDS) * len(FOLDS) * len(MODELS_TO_RUN)
def n_done():
    return sum(os.path.exists(f'{RUNS_DIR}/{m}_fold{f}_seed{s}/summary.json') for s in SEEDS for f in FOLDS for m in MODELS_TO_RUN)
def lr_check(run):
    h = pd.read_csv(f'{RUNS_DIR}/{run}/history.csv')
    col = 'learning_rate' if 'learning_rate' in h else 'lr'
    low = h.index[h[col] <= 1e-6]
    return (int(low[0]) + 1) if len(low) else None, len(h)
import time
print(f'already finished: {n_done()} / {N}')
for seed in SEEDS:
    for fold in FOLDS:
        for name in MODELS_TO_RUN:
            run = f'{name}_fold{fold}_seed{seed}'
            qseg2.run_fold(cfg, DATA, name, fold, seed, out_root=RUNS_DIR, min_epochs=15, verbose=0)
            shrink_weights(run)
            ep_low, n_ep = lr_check(run)
            print(time.strftime('%H:%M UTC', time.gmtime()), f'| {run}: {n_ep} epochs, lr<=1e-6 from epoch {ep_low} | done {n_done()} / {N}', flush=True)
print('all requested runs finished')


In [ ]:
# 5 · Auto-shutdown: save Drive, then release the GPU (runs only after cell 4 finishes)
import time
from google.colab import drive, runtime
print('training cell finished at', time.strftime('%H:%M:%S UTC', time.gmtime()))
drive.flush_and_unmount()
print('Drive flushed')
time.sleep(30)
runtime.unassign()
